# 🏴 Ozz — HALctf Agent
## DEF CON 34 AI Village — Autonomous Pentesting Agent

Este notebook roda no **Kaggle com GPU gratuita** para testar o agente.

### Pipeline:
1. Instalar dependências
2. Baixar modelo Qwen 2.5 Coder 7B
3. Iniciar servidor vLLM
4. Subir targets sintéticos (universo de teste)
5. Rodar o agente autônomo
6. Avaliar resultados

## 1. Setup — Instalar Dependências

In [ ]:
# Instalar vLLM e dependências
!pip install vllm requests pwntools beautifulsoup4 lxml pyjwt flask -q

# Instalar ferramentas de pentest
!apt-get update -qq && apt-get install -y -qq nmap nikto gobuster netcat-openbsd curl wget sqlmap hydra smbclient mysql-client -qq 2>/dev/null || echo 'Some tools may not be available'

In [ ]:
# Clonar o repositório
!git clone https://github.com/UNIFEI-CDA/MNHI-3.5.git /content/MNHI-3.5
%cd /content/MNHI-3.5/halctf-agent
!ls -la

## 2. Baixar Modelo

In [ ]:
# Baixar Qwen 2.5 Coder 7B Instruct
from huggingface_hub import snapshot_download

model_name = "Qwen/Qwen2.5-Coder-7B-Instruct"
model_path = "/content/models/qwen2.5-coder-7b"

print(f"📥 Baixando {model_name}...")
snapshot_download(
    repo_id=model_name,
    local_dir=model_path,
    local_dir_use_symlinks=False,
)
print("✅ Modelo baixado!")
!ls -la {model_path}

## 3. Iniciar Servidor vLLM

In [ ]:
# Iniciar vLLM em background
import subprocess
import time
import requests

vllm_proc = subprocess.Popen([
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", model_path,
    "--served-model-name", "qwen2.5-coder-7b",
    "--host", "0.0.0.0",
    "--port", "8000",
    "--gpu-memory-utilization", "0.85",
    "--max-model-len", "8192",
    "--trust-remote-code",
    "--dtype", "auto",
    "--enforce-eager",
], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Aguardar o servidor ficar pronto
print("⏳ Aguardando vLLM...")
for i in range(120):
    try:
        r = requests.get("http://localhost:8000/v1/models", timeout=2)
        if r.status_code == 200:
            print("✅ vLLM pronto!")
            print(f"Modelos: {r.json()}")
            break
    except:
        pass
    time.sleep(2)
    if i % 10 == 0:
        print(f"  Aguardando... ({i*2}s)")
else:
    print("❌ vLLM não iniciou a tempo")

## 4. Testar o LLM

In [ ]:
# Testar o modelo
import requests
import json

response = requests.post(
    "http://localhost:8000/v1/chat/completions",
    json={
        "model": "qwen2.5-coder-7b",
        "messages": [
            {"role": "system", "content": "You are a pentesting agent. Respond only with JSON."},
            {"role": "user", "content": "I found a web server running nginx 1.18 on port 80. What should I do next? Respond with JSON: {\"thought\": ..., \"action\": ..., \"action_input\": ...}"}
        ],
        "max_tokens": 500,
        "temperature": 0.3,
    },
    timeout=60,
)

data = response.json()
print("🧠 Resposta do modelo:")
print(json.dumps(data, indent=2, ensure_ascii=False))

## 5. Rodar Mock Test (Sem Targets Reais)

In [ ]:
# Mock test — valida o fluxo sem targets reais
!python3 scripts/mock_runner.py --scenario full --verbose

## 6. Rodar Agente com LLM Real

In [ ]:
# Importar e rodar o agente com o modelo real
import sys
sys.path.insert(0, '.')

from agent.llm import LLM
from agent.tools import ToolRegistry
from agent.memory import Memory

# Criar instâncias
llm = LLM(model_path=model_path, port=8000)
tools = ToolRegistry()
memory = Memory()

# Testar uma iteração
context = """You are Ozz, an autonomous pentesting agent.
Target: 10.0.0.10 (web server, nginx 1.18, PHP 7.4)
Phase: RECON
What's your first action?
Respond with JSON: {"thought": ..., "action": ..., "action_input": ...}"""

response = llm.generate(context)
print("🧠 LLM Response:")
print(response)

## 7. Avaliação

Métricas:
- Flags encontradas / total
- Tempo médio por flag
- Ações desperdiçadas (loops)
- Rollbacks necessários
- Pivot bem-sucedido

In [ ]:
# Relatório final
stats = memory.get_stats()
flags = memory.get_flags()

print("=" * 60)
print("🏴 OZZ — RELATÓRIO FINAL")
print("=" * 60)
print(f"Observações: {stats['observations']}")
print(f"Descobertas: {stats['findings']}")
print(f"Flags: {stats['flags']}")
print(f"Credenciais: {stats['credentials']}")
print()
for flag in flags:
    print(f"  🚩 {flag['flag']} (de {flag['source']})")
print("=" * 60)

In [ ]:
# Cleanup
vllm_proc.terminate()
print("🛑 vLLM encerrado.")